# Resident Evil Quality & Descriptive Checks

Simple QA pass over all CSVs
- File-level summary (rows, columns, file size)
- Missing values & duplicates
- Descriptive stats for numeric, boolean, and datetime columns
- Comment-quality checks: empty, very short, very long, non-text, duplicates, language distribution

In [ ]:
import os
import glob
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 120)

REVIEWS_DIR = '../raw_reviews'
files = sorted(glob.glob(os.path.join(REVIEWS_DIR, '*.csv')))
print(f'Found {len(files)} CSV file(s):')
for f in files:
    size_mb = os.path.getsize(f) / 1024 / 1024
    print(f'  {os.path.basename(f):60s} {size_mb:7.2f} MB')

## File-level summary

In [ ]:
summary_rows = []
for f in files:
    df = pd.read_csv(f, low_memory=False)
    summary_rows.append({
        'file': os.path.basename(f),
        'rows': len(df),
        'cols': df.shape[1],
        'size_mb': round(os.path.getsize(f) / 1024 / 1024, 2),
    })
summary_df = pd.DataFrame(summary_rows)
summary_df

## Per-file check

In [ ]:
def check_file(path):
    name = os.path.basename(path)
    print('=' * 90)
    print(f'FILE: {name}')
    print('=' * 90)

    df = pd.read_csv(path, low_memory=False)
    print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} cols')
    print(f'Columns: {list(df.columns)}\n')

    # --- Dtypes ---
    print('--- Dtypes ---')
    print(df.dtypes)
    print()

    # --- Missing values ---
    print('--- Missing values per column ---')
    miss = df.isna().sum()
    miss_pct = (miss / len(df) * 100).round(2)
    miss_tbl = pd.DataFrame({'n_missing': miss, 'pct_missing': miss_pct})
    print(miss_tbl[miss_tbl['n_missing'] > 0] if (miss > 0).any() else 'No missing values.')
    print()

    # --- Duplicate rows ---
    n_dup = df.duplicated().sum()
    print(f'--- Duplicate rows: {n_dup:,} ({n_dup/len(df)*100:.2f}%) ---')
    if 'recommendationid' in df.columns:
        n_dup_id = df['recommendationid'].duplicated().sum()
        print(f'    Duplicate recommendationid values: {n_dup_id:,}')
    print()

    # --- Numeric describe ---
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if num_cols:
        print('--- Numeric describe ---')
        print(df[num_cols].describe().T.round(2))
        print()

    # --- Boolean / categorical describe ---
    bool_like = [c for c in df.columns if df[c].dropna().isin([True, False, 0, 1, 'True', 'False']).all() and df[c].nunique(dropna=True) <= 2]
    for c in bool_like:
        print(f'--- Value counts: {c} ---')
        print(df[c].value_counts(dropna=False))
        print()

    # --- Datetime / timestamp columns ---
    ts_cols = [c for c in df.columns if 'timestamp' in c.lower() or c.lower().endswith('_at') or c.lower() in ('created', 'updated')]
    for c in ts_cols:
        s = df[c].dropna()
        if s.empty:
            continue
        # Steam timestamps are unix seconds
        if pd.api.types.is_numeric_dtype(s):
            try:
                dt = pd.to_datetime(s, unit='s', errors='coerce')
                print(f'--- Time range ({c}, treated as unix seconds) ---')
                print(f'  min: {dt.min()}   max: {dt.max()}')
                print()
            except Exception:
                pass

    # --- Language distribution (if present) ---
    if 'language' in df.columns:
        print('--- Language distribution (top 15) ---')
        print(df['language'].value_counts(dropna=False).head(15))
        print()

    # --- Comment-quality checks ---
    cc = detect_comment_col(df)
    if cc is None:
        print('!! No comment/review text column found — skipping text checks.')
        return df

    print(f'--- Comment quality checks (column: "{cc}") ---')
    text = df[cc].astype('string')
    n = len(text)
    n_null     = text.isna().sum()
    n_empty    = (text.fillna('').str.strip() == '').sum() - n_null  # empty-but-not-null
    n_blank    = n_null + n_empty
    lengths    = text.fillna('').str.len()
    n_short    = ((lengths > 0) & (lengths < SHORT_THRESHOLD)).sum()
    n_long     = (lengths > LONG_THRESHOLD).sum()
    n_dup_text = text.dropna().duplicated().sum()
    word_counts = text.fillna('').str.split().str.len()

    print(f'  total comments:          {n:,}')
    print(f'  null:                    {n_null:,} ({n_null/n*100:.2f}%)')
    print(f'  empty / whitespace-only: {n_empty:,} ({n_empty/n*100:.2f}%)')
    print(f'  total blank:             {n_blank:,} ({n_blank/n*100:.2f}%)')
    print(f'  shorter than {SHORT_THRESHOLD} chars:    {n_short:,} ({n_short/n*100:.2f}%)')
    print(f'  longer than {LONG_THRESHOLD} chars:  {n_long:,} ({n_long/n*100:.2f}%)')
    print(f'  exact duplicate texts:   {n_dup_text:,} ({n_dup_text/n*100:.2f}%)')
    print()
    print('  char length stats:')
    print(lengths.describe().round(2).to_string())
    print()
    print('  word count stats:')
    print(word_counts.describe().round(2).to_string())
    print()

    # Sample very short non-empty comments
    short_samples = df.loc[(lengths > 0) & (lengths < SHORT_THRESHOLD), cc].head(5).tolist()
    if short_samples:
        print('  sample very-short comments:')
        for s in short_samples:
            print(f'    - {s!r}')
        print()

    return df

In [ ]:
# Run checks across every file
for f in files:
    _ = check_file(f)
    print()

## 3. Cross-file comparison

Quick side-by-side: rows, blanks, duplicate texts, average comment length.

In [ ]:
rows = []
for f in files:
    df = pd.read_csv(f, low_memory=False)
    cc = detect_comment_col(df)
    rec = {'file': os.path.basename(f), 'n_rows': len(df), 'n_dup_rows': int(df.duplicated().sum())}
    if cc is not None:
        text = df[cc].astype('string')
        lengths = text.fillna('').str.len()
        rec.update({
            'comment_col': cc,
            'n_blank_comments': int(text.fillna('').str.strip().eq('').sum()),
            'n_dup_comments': int(text.dropna().duplicated().sum()),
            'mean_chars': round(lengths.mean(), 1),
            'median_chars': float(lengths.median()),
            'p95_chars': float(lengths.quantile(0.95)),
        })
    rows.append(rec)

compare_df = pd.DataFrame(rows)
compare_df